# 04 — Survival Analysis (Cox Proportional Hazards)

Estimates product survival probability given market conditions.
Model: $h(t|X) = h_0(t) \exp(\sum \beta_i X_i)$

**EN** — A Cox model relates market covariates to the *hazard* (instantaneous risk) that the traditional PBX market "dies". We report hazard ratios, survival curves and scenario survival probabilities.

**繁中** — Cox 比例風險模型將市場條件（寬頻、人均 GDP、都市化、PSTN 退場政策）連結到傳統 PBX 市場「消亡」的瞬時風險。本筆記本輸出風險比 (hazard ratio)、存活曲線與情境存活機率。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.models.survival import (
    fit_cox_model, plot_survival_curves, 
    predict_survival_probability, rank_markets,
    plot_hazard_ratios
)
from src.data.preprocessor import build_product_lifetime_table

In [ ]:
panel = pd.read_csv('data/processed/panel_data.csv')
print(f"Panel loaded: {panel.shape}")
print(f"Columns: {panel.columns.tolist()}")

## 4.1 Build Product Lifetime Table

Product: "Traditional PBX Gateway" — introduced in 2005.

### Why 2005 as the baseline (t = 0)? / 為什麼以 2005 年為基準（t = 0）？

**EN.** `product_intro_year = 2005` is the **assumed market-introduction year of the tracked product** ("Traditional PBX Gateway"). It defines the survival clock: `t = 0` at 2005, and every duration in this notebook is **years since 2005** (so `t = 3` → calendar 2008, `t = 5` → 2010, `t = 10` → 2015). We chose 2005 because:

1. **It is the start of the modern substitution era.** Broadband/VoIP adoption accelerated from the mid-2000s, which is precisely the decline pressure the covariates (broadband, PSTN phaseout) are meant to capture. Anchoring earlier would mix in the pre-broadband growth phase; anchoring later would discard most of the observable decline.
2. **Covariates are measured at t = 0.** The Cox model reads each market's broadband, GDP/capita, urbanisation and phaseout status **as of 2005**, so the baseline year and the covariate snapshot are the same point in time.
3. **It is a modelling assumption, not a data field.** Changing `product_intro_year` shifts the entire survival timeline; 2005 is a deliberate, documented choice, not an intrinsic property of the data.

**繁中.** `product_intro_year = 2005` 是**所追蹤產品（「傳統 PBX 閘道器」）假定的市場導入年**。它定義了存活時鐘：2005 年為 `t = 0`，本筆記本中所有存續時間皆為**自 2005 起算的年數**（故 `t = 3` → 西元 2008、`t = 5` → 2010、`t = 10` → 2015）。選擇 2005 的理由：

1. **它是現代替代浪潮的起點。** 寬頻／VoIP 自 2000 年代中期加速普及，正是共變量（寬頻、PSTN 退場）所要捕捉的衰退壓力。基準若更早，會混入寬頻前的成長期；若更晚，則會丟失大部分可觀測的衰退。
2. **共變量於 t = 0 量測。** Cox 模型讀取各市場**2005 年當時**的寬頻、人均 GDP、都市化與退場狀態，使基準年與共變量快照為同一時點。
3. **這是建模假設，而非資料欄位。** 變更 `product_intro_year` 會整體平移存活時間軸；2005 是刻意且有記錄的選擇，並非資料本身的固有屬性。

In [ ]:
product_intro_year = 2005
lifetime = build_product_lifetime_table(
    panel,
    product_intro_year=product_intro_year,
    penetration_col='fixed_subs_value',
    death_threshold=0.7,  # "dead" = penetration < 70% of peak (>=30% decline from peak). With REAL World Bank fixed-line data, stricter thresholds leave too few observed deaths to identify the Cox model (0.3 -> only 1 of 12 markets, non-convergent). 0.7 yields 9 of 12 observed deaths, enough to stably fit 4 covariates while still reflecting genuine market sunset. (The old 0.3 value was tuned to synthetic data and is invalid for the real series.)
)
print(f"Lifetime table: {lifetime.shape[0]} countries, events={int(lifetime['product_dead'].sum())}")
lifetime

**EN — How to read this table.** `market_lifetime` = years from product introduction (2005) until penetration falls below 5% of each country's historical peak. `product_dead=1` means the threshold was actually crossed within the data window; `product_dead=0` is *right-censored* (still alive at the last observed year). Covariates are taken at the introduction year.

**繁中 — 表格判讀。** `market_lifetime` 為自產品導入年 (2005) 起，到市場滲透率跌破各國歷史高峰 5% 為止的年數。`product_dead=1` 表示在資料期間內確實跨越門檻；`product_dead=0` 為*右設限*（在最後觀測年仍存活）。共變量取自導入年的數值。

## 4.2 Fit Cox Proportional Hazards Model

In [ ]:
covariates = ['broadband_value', 'gdp_per_capita_value', 
              'urban_pop_value', 'has_pstn_phaseout']
available_covs = [c for c in covariates if c in lifetime.columns]
print(f"Using covariates: {available_covs}")

model = fit_cox_model(
    lifetime,
    duration_col='market_lifetime',
    event_col='product_dead',
    covariates=available_covs,
    penalizer=0.1,
)
print("CoxPH model fitted successfully.")
print(f"N = {lifetime.shape[0]} countries, events (deaths) = {int(lifetime['product_dead'].sum())}")
model.print_summary()

**EN — Caveat on sample size.** This model is fitted on a small panel (~13 countries). With 4 covariates and a ridge `penalizer=0.1`, coefficient estimates are stabilised but confidence intervals are wide and p-values should be read as *directional*, not confirmatory. Covariates are z-score standardized internally, so each hazard ratio is per +1 standard deviation.

**繁中 — 樣本數警語。** 本模型僅以約 13 個國家的小樣本配適。使用 4 個共變量並加上 ridge 懲罰項 (`penalizer=0.1`) 雖能穩定係數，但信賴區間偏寬、p 值僅供*方向性*參考。共變量在模型內部已 z 分數標準化，故每個風險比代表每增加 1 個標準差的效果。

## 4.3 Hazard Ratios

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_hazard_ratios(model, ax=ax)
plt.tight_layout()
plt.savefig('data/processed/hazard_ratios.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nInterpretation:")
print("  HR > 1 = higher risk (product dies sooner)")
print("  HR < 1 = lower risk (product survives longer)")

**EN.** A hazard ratio (HR) of 1 means no effect. HR > 1 → that covariate accelerates market death (shorter survival); HR < 1 → it prolongs survival. The horizontal bars are 95% confidence intervals on the HR scale; bars crossing the dashed line at 1.0 are not statistically distinguishable from "no effect".

**繁中.** 風險比 (HR) 等於 1 表示無影響。HR > 1 → 該變量加速市場消亡（存活較短）；HR < 1 → 延長存活。水平線段為 HR 尺度上的 95% 信賴區間；跨越 1.0 虛線者，與「無影響」在統計上無法區分。

## 4.4 Survival Curves by Country

In [ ]:
countries = lifetime['country'].tolist()
fig, ax = plt.subplots(figsize=(10, 6))
plot_survival_curves(model, countries, lifetime, t_max=20, ax=ax)
plt.tight_layout()
plt.savefig('data/processed/survival_curves.png', dpi=150, bbox_inches='tight')
plt.show()

**EN.** Each curve $S(t)$ is the model-estimated probability that the PBX market in that country is still "alive" $t$ years after introduction, given its covariates. Curves that fall faster indicate markets expected to sunset sooner. Beyond the largest observed lifetime the curve is *extrapolated* (held flat) — treat the far tail with caution.

**繁中.** 每條曲線 $S(t)$ 為模型估計：在給定共變量下，該國 PBX 市場於導入後第 $t$ 年仍「存活」的機率。下降越快代表預期越早退場。超過最大觀測壽命的部分屬於*外推*（維持水平），尾端需謹慎解讀。

## 4.5 Market Rankings

In [ ]:
for t in [3, 5, 10]:
    rankings = rank_markets(model, lifetime, t=t)
    print(f"\n=== Market Rankings: {t}-year Survival Probability ===")
    print(rankings.to_string(index=False))

**EN.** Countries are ranked by $S(t)$: higher = the traditional PBX market is expected to persist longer (more time to monetise/maintain). These rankings use the **same decline-based definition of market death** as notebook 03, so the two notebooks are consistent.

**繁中.** 依 $S(t)$ 排名：數值越高代表傳統 PBX 市場預期持續越久（可維運/變現的時間越長）。此排名採用與筆記本 03 **相同的、以衰退期定義的市場消亡**標準，故兩份筆記本結論一致。

## 4.6 Scenario: Product Launch Decision (East/SE Asia, Taiwan, India, JP/KR vs. PSTN-phaseout West)

**EN.** Reference markets requested: East/SE Asia + Taiwan + India + Japan/South Korea, contrasted with Western markets that have an **announced PSTN phaseout** (Germany, UK, Sweden) and high-broadband hubs (Hong Kong, Singapore). Covariates are on the **2005 at-introduction scale** the model was trained on (broadband ~4–15, gdp ~30–34k, urban ~65–68); present-day values would be out-of-distribution. "Death" is defined as penetration falling below 30% of peak (≈70% decline), so survival here means *the traditional PBX market is still the primary access technology*.

**繁中.** 依需求的參考市場：東亞/東南亞＋台灣＋印度＋日本/南韓，並與**已宣布 PSTN 退場**的西方市場（德國、英國、瑞典）及高寬頻樞紐（香港、新加坡）對比。共變量採模型訓練所用的 **2005 導入年尺度**（寬頻約 4–15、人均 GDP 約 30–34k、都市化約 65–68）；現今實際值屬分布外。「消亡」定義為滲透率跌破高峰 30%（≈70% 衰退），故此處存活代表*傳統 PBX 市場仍為主要接取技術*。

In [ ]:
# 4.6 Scenario: Product Launch Decision.
# Focus per request: East/SE Asia + Taiwan + India + Japan/South Korea, vs.
# Western markets with an announced PSTN phaseout (Germany, UK, Sweden).
# Covariates are 2005 at-introduction values (the scale the model was trained
# on); has_pstn_phaseout=1 flags an announced legacy switch-off.
scenarios = [
    {"name": "Taiwan",                       "covs": {'broadband_value': 9.5,  'gdp_per_capita_value': 32567, 'urban_pop_value': 65.1, 'has_pstn_phaseout': 0}},
    {"name": "India (low broadband)",        "covs": {'broadband_value': 7.2,  'gdp_per_capita_value': 30797, 'urban_pop_value': 66.4, 'has_pstn_phaseout': 0}},
    {"name": "China",                        "covs": {'broadband_value': 8.6,  'gdp_per_capita_value': 33475, 'urban_pop_value': 66.4, 'has_pstn_phaseout': 0}},
    {"name": "Hong Kong",                    "covs": {'broadband_value': 13.0, 'gdp_per_capita_value': 33500, 'urban_pop_value': 67.5, 'has_pstn_phaseout': 0}},
    {"name": "Singapore (phaseout)",         "covs": {'broadband_value': 12.0, 'gdp_per_capita_value': 33000, 'urban_pop_value': 67.9, 'has_pstn_phaseout': 1}},
    {"name": "Japan (phaseout)",             "covs": {'broadband_value': 14.4, 'gdp_per_capita_value': 31509, 'urban_pop_value': 65.9, 'has_pstn_phaseout': 1}},
    {"name": "South Korea",                  "covs": {'broadband_value': 11.4, 'gdp_per_capita_value': 31770, 'urban_pop_value': 66.7, 'has_pstn_phaseout': 0}},
    {"name": "Germany (phaseout)",           "covs": {'broadband_value': 9.5,  'gdp_per_capita_value': 29803, 'urban_pop_value': 66.4, 'has_pstn_phaseout': 1}},
    {"name": "United Kingdom (phaseout 2027)","covs": {'broadband_value': 7.7,  'gdp_per_capita_value': 33030, 'urban_pop_value': 67.9, 'has_pstn_phaseout': 1}},
    {"name": "Sweden (phaseout)",            "covs": {'broadband_value': 12.4, 'gdp_per_capita_value': 33921, 'urban_pop_value': 65.9, 'has_pstn_phaseout': 1}},
    {"name": "USA",                          "covs": {'broadband_value': 4.3,  'gdp_per_capita_value': 33649, 'urban_pop_value': 64.8, 'has_pstn_phaseout': 0}},
]

print("=== Scenario: Product Launch Decision ===")
rows = []
for sc in scenarios:
    # Keep only covariates used by the fitted model; standardized internally.
    filtered_covs = {k: v for k, v in sc['covs'].items() if k in available_covs}
    s3 = predict_survival_probability(model, filtered_covs, t=3)
    s5 = predict_survival_probability(model, filtered_covs, t=5)
    s10 = predict_survival_probability(model, filtered_covs, t=10)
    rows.append({'scenario': sc['name'], 's3y': s3, 's5y': s5, 's10y': s10})
scenario_df = pd.DataFrame(rows).sort_values('s10y', ascending=False)
for _, r in scenario_df.iterrows():
    print(f"\n{r['scenario']}")
    print(f"  3-year survival:  {r['s3y']:.1%}")
    print(f"  5-year survival:  {r['s5y']:.1%}")
    print(f"  10-year survival: {r['s10y']:.1%}")

**EN — Scenario reading.** Markets are sorted by 10-year survival. With the corrected death definition (penetration < 30% of peak), survival now **decays toward near-zero by year 10** for advanced markets — consistent with §4.5 rankings, notebook 03's early-2010s 70%-decline death years, and the overall "legacy PBX is sunsetting" conclusion (no longer the old, decoupled 80–97% flat line). The spread is driven by the §4.3 hazard ratios: an announced **PSTN phaseout** and **higher early broadband** raise the death hazard, so Germany/UK/Sweden/Japan sunset fastest, while lower-broadband, no-phaseout markets (India, USA, China, Taiwan) retain a slightly longer legacy window. Use the 3-year figure for near-term viability and the 10-year figure for end-of-life planning.

**繁中 — 情境判讀。** 依 10 年存活率排序。採用修正後的消亡定義（滲透率 < 高峰 30%）後，先進市場的存活率**到第 10 年已趨近於零**——與 §4.5 排名、筆記本 03 的 2010 年代初期 70% 衰退消亡年、以及「傳統 PBX 正在退場」的整體結論一致（不再是舊有、脫鉤的 80–97% 水平線）。差異由 §4.3 風險比驅動：已宣布 **PSTN 退場**與**早期寬頻較高**會提高消亡風險，故德國/英國/瑞典/日本最快退場，而寬頻較低、未退場的市場（印度、美國、中國、台灣）保有略長的傳統時間窗。短期可行性看 3 年值，汰換/退役規劃看 10 年值。

## 4.7 Long-horizon projection to 2045 (科技奇異點) / 長期展望至 2045（科技奇異點）

**EN — Why not just extend §4.6 to 2027–2045?** The Cox survival model in §4.6 is **horizon-limited**: its baseline hazard is only estimated out to the longest *observed* lifetime (~8 years from 2005 ≈ 2013). Asking it for 2027–2045 returns the **last value, held flat** — uninformative. For genuine calendar-year projection we instead use each market's **logistic decline curve**, a parametric model that legitimately extrapolates. The metric below is **penetration as % of each market's historical peak** (a "fraction-of-market-remaining" survival analogue). 2045 is included as the commonly-cited *technological-singularity* horizon — treat anything past the data end (2025) as a model extrapolation under deep uncertainty.

**繁中 — 為何不直接把 §4.6 延伸到 2027–2045？** §4.6 的 Cox 存活模型**受時間視野限制**：其基準風險僅估計到最長*觀測*壽命（自 2005 約 8 年 ≈ 2013）。要求其輸出 2027–2045 只會回傳**最後一個值並維持水平**，不具資訊量。要做真正的日曆年展望，改用各市場的**邏輯斯衰退曲線**（可正當外推的參數模型）。下方指標為**滲透率占各市場歷史高峰的百分比**（「市場剩餘比例」的存活類比）。納入 2045 作為常被引用的*科技奇異點*視野——資料結束（2025）之後皆屬深度不確定下的模型外推。

In [ ]:
from src.models.logistic_growth import fit_all_countries, logistic_decline

# §4.6 (Cox) cannot reach 2027-2045 (baseline ends ~2013). Use the per-country
# logistic DECLINE curve, which extrapolates, to project calendar-year market size.
decline_fits = fit_all_countries(panel, penetration_col='fixed_subs_value', death_threshold=0.3)

horizon_years = [2027, 2028, 2029, 2030, 2035, 2040, 2045]  # 2045 ≈ posited tech singularity
last_data_year = int(panel['year'].max())
focus = ['tw', 'jp', 'kr', 'cn', 'in', 'de', 'gb', 'se', 'us']  # East/SE Asia + JP/KR vs PSTN-phaseout West

rows = []
for code in focus:
    sub = decline_fits[decline_fits['country'] == code]
    if sub.empty:
        continue
    r = sub.iloc[0]
    if not r['converged'] or pd.isna(r['decline_K']):
        continue
    rec = {'market': code.upper()}
    for yr in horizon_years:
        pct = logistic_decline(np.array([float(yr)]), r['decline_K'], r['decline_r'], r['decline_t0'])[0]
        rec[f'{yr}'] = round(max(0.0, pct / r['peak_value'] * 100), 2)
    rows.append(rec)

projection = pd.DataFrame(rows).set_index('market')
print("Traditional fixed-line / PBX penetration projected as % of each market's HISTORICAL PEAK.")
print(f"Historical data ends {last_data_year}; ALL columns below are model EXTRAPOLATION beyond it.\n")
print(projection.to_string())
projection.to_csv('data/processed/longhorizon_projection.csv')
print("\nSaved to data/processed/longhorizon_projection.csv")

**EN — Reading the long-horizon table.** Each cell is the modelled traditional fixed-line/PBX penetration **as a % of that market's historical peak** in the given calendar year. The story is unambiguous: every focus market is already at only a **few percent of peak by 2027** and asymptotes toward **near-zero by 2045** (the posited technological-singularity horizon). South Korea, Sweden, the UK and the USA retain marginally more legacy base than India, Germany or Taiwan, but all are effectively sunset long before 2045. **Practical takeaway:** the traditional PBX/PSTN market is a *runoff* business from now on — there is no future-year window in which it recovers, so investment should target the IP/API and non-network replacement families (see notebooks 07 and the frontend catalog), and migrations should be planned against the §4.6 near-term (3-year) survival, not the long tail. Remember all 2026+ figures are extrapolations of a curve fit to 2000–2025 data and carry deep uncertainty (technology, regulation, and singularity-era shocks are not modelled).

**繁中 — 長期展望表判讀。** 每格為模型估計之傳統固網/PBX 滲透率，以**占該市場歷史高峰的百分比**表示於指定日曆年。結論明確：所有焦點市場**到 2027 年都僅剩高峰的數個百分點**，並在**2045 年（所設定的科技奇異點視野）趨近於零**。南韓、瑞典、英國與美國的傳統基數略多於印度、德國或台灣，但全部都在 2045 之前早已退場。**實務啟示：** 傳統 PBX/PSTN 市場自此為*收尾（runoff）*業務——不存在任何未來年度會回升的時間窗，故投資應鎖定 IP/API 與非網路替代類別（見筆記本 07 與前端目錄），而遷移時程應依 §4.6 的近期（3 年）存活規劃，而非長尾。請注意所有 2026 年以後數值皆為以 2000–2025 資料配適之曲線的外推，具深度不確定性（技術、法規與奇異點時代的衝擊皆未納入模型）。